# RAG Components with Groq and Local Embeddings

This notebook builds each part of a modern **Retrieval-Augmented Generation (RAG)** pipeline and then connects the parts with LangChain Expression Language (LCEL).

## What you will learn

1. Load a source into LangChain `Document` objects.
2. Split long text into overlapping chunks.
3. Convert chunks into embeddings locally.
4. Store and search embeddings with Chroma.
5. Expose vector search through a retriever.
6. Send retrieved context to a Groq chat model.

```text
Source → Documents → Chunks → Embeddings → Vector store
                                                ↓
Question → Retriever → Relevant chunks → Prompt → Groq → Answer
```

> Select the **Python 3.13.7 (`.venv`)** kernel and run the cells from top to bottom.

## 1. Production-style configuration

### What?

Configuration controls model names, chunk sizes, retrieval count, and output limits.

### Why?

Enterprise applications keep tunable values outside business logic. A typed, immutable settings object validates values early and prevents accidental changes during execution.

Add these values to `.env` if you want to override the defaults:

```text
GROQ_API_KEY=your_groq_api_key
GROQ_MODEL=qwen/qwen3.8-27b
GROQ_MAX_TOKENS=256
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2
RAG_CHUNK_SIZE=800
RAG_CHUNK_OVERLAP=120
RAG_TOP_K=3
```

Never commit a real API key.

In [1]:
import logging
import os
from dataclasses import dataclass
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("rag_components")


def positive_int_from_env(name: str, default: int) -> int:
    raw_value = os.getenv(name, str(default))
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be an integer, received {raw_value!r}.") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero.")
    return value


@dataclass(frozen=True, slots=True)
class Settings:
    groq_model: str
    embedding_model: str
    max_output_tokens: int
    chunk_size: int
    chunk_overlap: int
    top_k: int

    @classmethod
    def from_environment(cls) -> "Settings":
        if not os.getenv("GROQ_API_KEY"):
            raise RuntimeError("GROQ_API_KEY is missing from the .env file.")

        settings = cls(
            groq_model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
            embedding_model=os.getenv(
                "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
            ),
            max_output_tokens=positive_int_from_env("GROQ_MAX_TOKENS", 256),
            chunk_size=positive_int_from_env("RAG_CHUNK_SIZE", 800),
            chunk_overlap=positive_int_from_env("RAG_CHUNK_OVERLAP", 120),
            top_k=positive_int_from_env("RAG_TOP_K", 3),
        )
        if settings.chunk_overlap >= settings.chunk_size:
            raise ValueError("RAG_CHUNK_OVERLAP must be smaller than RAG_CHUNK_SIZE.")
        return settings


settings = Settings.from_environment()
logger.info("Configuration loaded for Groq model %s", settings.groq_model)

2026-10-01 10:08:14,257 | INFO | rag_components | Configuration loaded for Groq model qwen/qwen3.8-27b


## 2. Load a document

### What is a LangChain `Document`?

A `Document` contains:

| Field | Purpose |
| --- | --- |
| `page_content` | Text used for splitting, embedding, retrieval, and prompting |
| `metadata` | Source information such as a file path, page, or chunk number |

### When and why?

Convert every input source to the same `Document` format. Downstream components can then work consistently with text files, PDFs, APIs, or databases.

This example uses `Path.read_text()` instead of the sunset `langchain-community` text loader.

In [2]:
from langchain_core.documents import Document


def load_text_document(path: str | Path) -> list[Document]:
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"Required source file was not found: {file_path.resolve()}")

    content = file_path.read_text(encoding="utf-8").strip()
    if not content:
        raise ValueError(f"Source file is empty: {file_path}")

    logger.info("Loaded %d characters from %s", len(content), file_path)
    return [Document(page_content=content, metadata={"source": str(file_path)})]


source_documents = load_text_document("data/state_of_the_union.txt")
print("Documents:", len(source_documents))
print("Metadata:", source_documents[0].metadata)
print("Preview:\n", source_documents[0].page_content[:400])

2026-10-01 10:08:24,075 | INFO | rag_components | Loaded 38539 characters from data\state_of_the_union.txt


Documents: 1
Metadata: {'source': 'data\\state_of_the_union.txt'}
Preview:
 Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Con


## 3. Split text into chunks

### What is chunking?

Chunking divides a long document into smaller pieces. Embedding and retrieving focused pieces is more useful than treating a whole large file as one result.

### Important parameters

| Parameter | Meaning | Why it matters |
| --- | --- | --- |
| `chunk_size` | Maximum approximate characters per chunk | Larger chunks carry more context but may contain unrelated details |
| `chunk_overlap` | Characters repeated between neighboring chunks | Helps preserve meaning across a split boundary |
| `separators` | Preferred boundaries such as paragraphs and sentences | Avoids cutting text in unnatural places |

### Character versus recursive splitting

`CharacterTextSplitter` uses one separator. `RecursiveCharacterTextSplitter` tries several separators from strongest to weakest and is usually the safer general-purpose choice.

In [6]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

character_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    length_function=len,
)
character_chunks = character_splitter.split_documents(source_documents)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    length_function=len,
    add_start_index=True,
)
chunks = recursive_splitter.split_documents(source_documents)

for chunk_number, chunk in enumerate(chunks):
    chunk.metadata["chunk_number"] = chunk_number

logger.info(
    "Created %d character chunks and %d recursive chunks",
    len(character_chunks),
    len(chunks),
)
print("Character splitter chunks:", len(character_chunks))
print("Recursive splitter chunks:", len(chunks))

print("Character splitter chunks:")
for index, chunk in enumerate(character_chunks, start=1):
    print(f"\n---Character splitter Chunk {index} ---")
    print(chunk.page_content)

print("\nRecursive splitter chunks:")
for index, chunk in enumerate(chunks, start=1):
    print(f"\n---Recursive splitter Chunk {index} ---")
    print(chunk.page_content)

2026-10-01 10:20:51,139 | INFO | rag_components | Created 57 character chunks and 57 recursive chunks


Character splitter chunks: 57
Recursive splitter chunks: 57
Character splitter chunks:

---Character splitter Chunk 1 ---
Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined.

---Character splitter Chunk 2 ---
He met the Ukrainian people. 

From President Zelenskyy t

In [9]:
first_chunk = chunks[0]
print("First chunk metadata:", first_chunk.metadata)
print("First chunk length:", len(first_chunk.page_content))
print("First chunk preview:\n", first_chunk.page_content[:500])

First chunk metadata: {'source': 'data\\state_of_the_union.txt', 'start_index': 0, 'chunk_number': 0}
First chunk length: 785
First chunk preview:
 Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six day


## 4. Create embeddings locally

### What is an embedding?
Transform the small parts of text in numbers(vectors) that are easily stored and searched by vector databases.

An embedding is a list of numbers representing the semantic meaning of text. Texts with similar meaning should have vectors that are close together.

### When and why?

Embeddings power semantic search. Unlike keyword matching, semantic search can match different wording with similar meaning.

### Why not use Groq for embeddings?

Groq hosts chat/generation models but does not provide an embedding API. This notebook uses `HuggingFaceEmbeddings` with a local Sentence Transformers model, so embeddings do not consume OpenAI or Groq API credit.

> The first run downloads the embedding model. Later runs use the local cache.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name=settings.embedding_model,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

example_texts = [
    "The president discussed voting rights.",
    "The speech mentioned protecting access to elections.",
    "A recipe needs flour and water.",
]
example_vectors = embeddings_model.embed_documents(example_texts)

print("Number of vectors:", len(example_vectors))
print("Dimensions per vector:", len(example_vectors[0]))
print("First five values:", example_vectors[0][:5])

2026-10-01 10:49:26,843 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 10:49:26,847 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-01 10:49:26,895 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-01 10:49:27,174 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 10:49:27,224 | INFO | httpx | HTTP Request: HEAD htt

Number of vectors: 3
Dimensions per vector: 384
First five values: [-0.0010640228865668178, 0.002869404386729002, -0.025134827941656113, 0.06061010807752609, -0.04111329838633537]


## 5. Store embeddings in Chroma

### What is a vector store?
Store embeddings in a very fast searchable database.

A vector store saves text chunks, metadata, and embedding vectors. It performs fast similarity searches using a question's embedding.

### When and why?

Use a vector store when your knowledge base is too large to place entirely in an LLM prompt.

This lesson uses an in-memory Chroma collection. In production, configure persistent storage or a managed vector database, use stable document IDs, and apply tenant/access filters.

## Free / Open-Source (Self-Hosted)

* Chroma
* FAISS
* Milvus
* pgvector
* Qdrant
* Weaviate
* OpenSearch

## Paid / Managed Cloud & Enterprise

* Pinecone
* Zilliz Cloud
* Weaviate Cloud
* Qdrant Cloud
* Redis Cloud



In [ ]:
from langchain_chroma import Chroma
# choma db is a vector database used to store and retrieve embeddings efficiently.
# Here, we are creating a Chroma vector store from the document chunks and the embeddings model.
# it is in-memory by default, but can be configured to persist data to disk.

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="rag_components_demo",
)
logger.info("Indexed %d chunks in Chroma", len(chunks))
# %d is a placeholder for an integer value in the log message. 
# Here, it will be replaced by the number of chunks indexed in Chroma.

2026-10-01 11:08:40,940 | INFO | rag_components | Indexed 57 chunks in Chroma


In [10]:
search_question = "What did the president say about the John Lewis Voting Rights Act?"
search_results = vector_store.similarity_search(search_question, k=settings.top_k)

for rank, document in enumerate(search_results, start=1):
    print(f"Result {rank} | metadata={document.metadata}")
    print(document.page_content[:350], "\n")

Result 1 | metadata={'source': 'data\\state_of_the_union.txt', 'chunk_number': 42, 'start_index': 28635}
The most fundamental right in America is the right to vote – and to have it counted. And it’s under assault. 

In state after state, new laws have been passed, not only to suppress the vote, but to subvert entire elections. 

We cannot let this happen. 

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Righ 

Result 2 | metadata={'chunk_number': 45, 'source': 'data\\state_of_the_union.txt', 'start_index': 30747}
Provide a pathway to citizenship for Dreamers, those on temporary status, farm workers, and essential workers. 

Revise our laws so businesses have the workers they need and families don’t wait decades to reunite. 

It’s not only the right thing to do—it’s the economically smart thing to do. 

That’s why immigration reform is supported by everyone  

Result 3 | metadata={'start_index': 788, 'source': 'data\\state_of_the_union.txt', '

## 6. Use the vector store as a retriever
Find the embedding that best answers your question.

### What is a retriever?

A retriever accepts a question and returns relevant `Document` objects. It provides a standard LangChain interface over vector search.

### When and why?

Use a retriever when connecting search to an LCEL chain. The rest of the application does not need to know whether retrieval uses Chroma, another database, or a custom search service.

`k` controls how many chunks are returned. Too few may miss evidence; too many may add noise and consume prompt tokens.

In [ ]:
#as retriever from the Chroma vector store because it allows us to efficiently search for relevant document chunks based on their embeddings.

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": settings.top_k},
)
#top_k specifies the number of top similar documents to retrieve from the vector store.

retrieved_documents = retriever.invoke(
    "What did the president say about Ketanji Brown Jackson?"
)
print("Retrieved documents:", len(retrieved_documents))
print("Top result metadata:", retrieved_documents[0].metadata)
print("Top result preview:\n", retrieved_documents[0].page_content[:500])

Retrieved documents: 3
Top result metadata: {'start_index': 29345, 'chunk_number': 43, 'source': 'data\\state_of_the_union.txt'}
Top result preview:
 One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence. 

A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. 


## 7. Build the Groq generation model

The **embedding model** represents text for search. The **chat model** reads retrieved context and writes the final answer. They are separate components.

For factual RAG, a low temperature makes output more consistent. `max_tokens` limits answer length and cost.

In [ ]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(
    model=settings.groq_model,
    temperature=0.1,
    reasoning_effort="none",
    max_tokens=settings.max_output_tokens,
    max_retries=2,
    timeout=30,
)

## 8. Assemble the complete RAG chain with LCEL

### What is LCEL?

**LangChain Expression Language (LCEL)** uses `|` to pass one component's output to the next.

### What happens here?

1. `RunnableParallel` sends the question to the retriever and also preserves the original question.
2. Retrieved `Document` objects are formatted into labeled context.
3. `ChatPromptTemplate` instructs Groq to use only that context.
4. `StrOutputParser` returns a normal Python string.

> **Privacy:** The final invocation sends retrieved excerpts from the local source document to Groq. Run it only when you are authorized to share that data with the configured provider.

```text
question ─┬─→ retriever → format documents ─→ context ─┐
          └─→ pass through ────────────────→ question ├→ prompt → Groq → string
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


def format_documents(documents: list[Document]) -> str:
    if not documents:
        return "No relevant context was retrieved."

    formatted_sections = []
    for index, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "unknown")
        chunk_number = document.metadata.get("chunk_number", "unknown")
        formatted_sections.append(
            f"[Source {index}: {source}, chunk {chunk_number}]\n{document.page_content}"
        )
    return "\n\n".join(formatted_sections)


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a factual question-answering assistant. "
            "Use only the supplied context. If the context does not contain the answer, "
            "say that you do not know from the provided context. "
            "Do not invent facts. Mention the source numbers supporting the answer.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)

rag_chain = (
    RunnableParallel(
        context=retriever | format_documents,
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

In [ ]:
def ask_rag(question: str) -> str:
    clean_question = question.strip()
    if not clean_question:
        raise ValueError("Question cannot be empty.")

    logger.info("Running RAG query with top_k=%d", settings.top_k)
    return rag_chain.invoke(clean_question).strip()


answer = ask_rag("What did the president say about Ketanji Brown Jackson?")
print(answer)

## 9. Production and enterprise checklist

This notebook demonstrates production-minded structure, but a deployed system needs additional controls.

| Area | Notebook practice | Production extension |
| --- | --- | --- |
| Secrets | API key comes from `.env` | Use a managed secret store and rotate keys |
| Configuration | Typed and validated settings | Use environment-specific configuration management |
| Observability | Structured stage logging | Add tracing, metrics, latency, token usage, and alerting |
| Reliability | Model retries and timeout | Add rate-limit handling, circuit breakers, and provider fallback |
| Data | Source and chunk metadata | Add stable IDs, document versions, deletion, and re-indexing |
| Security | Local trusted file | Validate uploads, enforce access filters, and defend against prompt injection |
| Retrieval | Similarity search with `k` | Evaluate chunking, search type, reranking, and relevance thresholds |
| Quality | Context-only prompt | Build retrieval and answer evaluation datasets |

### Main idea

RAG quality depends on the entire pipeline—not only the LLM. Poor loading, chunking, embeddings, or retrieval cannot be reliably repaired by a stronger chat model.